In [ ]:
import time
import random
import math
import psutil
import os

In [ ]:
def memory_usage():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024  # MB

In [ ]:
def is_safe(board, col, row):
    for i in range(col):
        if board[i] == row or abs(board[i] - row) == abs(i - col):
            return False
    return True

In [ ]:
def dfs_n_queens(N):

    board = [-1] * N

    # tracking arrays
    rows = [False] * N
    diag1 = [False] * (2 * N)      # r - c + N
    diag2 = [False] * (2 * N)      # r + c

    def solve(col):
        if col == N:
            return True

        for row in range(N):

            if rows[row] or diag1[row - col + N] or diag2[row + col]:
                continue

            # place queen
            board[col] = row
            rows[row] = True
            diag1[row - col + N] = True
            diag2[row + col] = True

            if solve(col + 1):
                return True

            # backtrack
            rows[row] = False
            diag1[row - col + N] = False
            diag2[row + col] = False

        return False

    if solve(0):
        return board

    return None

In [ ]:
def cost(board):
    attacks = 0
    n = len(board)
    for i in range(n):
        for j in range(i + 1, n):
            if board[i] == board[j] or abs(board[i] - board[j]) == abs(i - j):
                attacks += 1
    return attacks

In [ ]:
def random_board(N):
    return [random.randint(0, N - 1) for _ in range(N)]

In [ ]:
import random

def hill_climbing(N, max_steps=1000, restarts=10):

    def init_board():
        return [random.randint(0, N - 1) for _ in range(N)]

    def compute_conflicts(board):
        row = [0]*N
        diag1 = [0]*(2*N)
        diag2 = [0]*(2*N)

        for c in range(N):
            r = board[c]
            row[r] += 1
            diag1[r - c + N] += 1
            diag2[r + c] += 1

        return row, diag1, diag2

    def conflicts_at(board, col, row, row_count, d1, d2):
        return (
            row_count[row] +
            d1[row - col + N] +
            d2[row + col]
        )

    for _ in range(restarts):

        board = init_board()
        row_count, d1, d2 = compute_conflicts(board)

        for step in range(max_steps):

            # check if solved
            total_conflicts = 0
            for c in range(N):
                r = board[c]
                total_conflicts += (
                    row_count[r] +
                    d1[r - c + N] +
                    d2[r + c] - 3
                )

            if total_conflicts == 0:
                return board

            # pick a random column with conflict
            conflict_cols = []
            for c in range(N):
                r = board[c]
                if (row_count[r] + d1[r - c + N] + d2[r + c]) > 3:
                    conflict_cols.append(c)

            if not conflict_cols:
                break

            col = random.choice(conflict_cols)

            # find best row in this column
            min_conflict = float('inf')
            best_rows = []

            for r in range(N):
                conf = conflicts_at(board, col, r, row_count, d1, d2)

                if conf < min_conflict:
                    min_conflict = conf
                    best_rows = [r]
                elif conf == min_conflict:
                    best_rows.append(r)

            new_row = random.choice(best_rows)
            old_row = board[col]

            # update counts
            row_count[old_row] -= 1
            d1[old_row - col + N] -= 1
            d2[old_row + col] -= 1

            board[col] = new_row

            row_count[new_row] += 1
            d1[new_row - col + N] += 1
            d2[new_row + col] += 1

    return None

In [ ]:
def simulated_annealing(N, T=1000, cooling=0.99, min_T=0.1):
    current = random_board(N)

    while T > min_T:
        current_cost = cost(current)

        if current_cost == 0:
            return current

        col = random.randint(0, N - 1)
        row = random.randint(0, N - 1)

        new_state = current[:]
        new_state[col] = row

        delta = cost(new_state) - current_cost

        if delta < 0 or random.random() < math.exp(-delta / T):
            current = new_state

        T *= cooling

    return current

In [ ]:
def fitness(board):
    n = len(board)
    non_attacks = 0
    total_pairs = n * (n - 1) // 2

    for i in range(n):
        for j in range(i + 1, n):
            if board[i] != board[j] and abs(board[i] - board[j]) != abs(i - j):
                non_attacks += 1

    return non_attacks

In [ ]:
def create_population(pop_size, N):
    return [[random.randint(0, N - 1) for _ in range(N)] for _ in range(pop_size)]

In [ ]:
def select_population(pop):
    weights = [fitness(ind) for ind in pop]
    total = sum(weights)
    probs = [w / total for w in weights]
    return random.choices(pop, weights=probs, k=len(pop)//2)

In [ ]:
def crossover(p1, p2):
    n = len(p1)
    point = random.randint(1, n - 2)
    return p1[:point] + p2[point:]

In [ ]:
def mutate(child, mutation_rate=0.1):
    for i in range(len(child)):
        if random.random() < mutation_rate:
            child[i] = random.randint(0, len(child) - 1)
    return child

In [ ]:
def genetic_algorithm(N, pop_size=50, generations=500, mutation_rate=0.1):

    population = create_population(pop_size, N)

    for _ in range(generations):

        if len(population) == 0:
            population = create_population(pop_size, N)

        population.sort(key=fitness, reverse=True)

        # perfect solution found
        if fitness(population[0]) == N*(N-1)//2:
            return population[0]

        selected = select_population(population)

        if len(selected) < 2:
            selected = create_population(pop_size, N)

        next_gen = []

        for i in range(0, len(selected)-1, 2):
            p1 = selected[i]
            p2 = selected[i+1]

            child = crossover(p1, p2)
            child = mutate(child, mutation_rate)
            next_gen.append(child)
            
        if len(next_gen) == 0:
            next_gen = create_population(pop_size, N)

        population = next_gen

    return max(population, key=fitness)

In [ ]:
def run_algo(name, func, N):
    print(f"\nRunning {name} for N={N}")

    start_mem = memory_usage()
    start_time = time.time()

    result = func(N)

    end_time = time.time()
    end_mem = memory_usage()

    print("Result found:", result is not None)
    print("Time:", end_time - start_time, "sec")
    print("Memory:", end_mem - start_mem, "MB")

    return end_time - start_time, end_mem - start_mem

In [ ]:
Ns = [10, 30, 50, 100, 200, 500]
#Ns = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30]
algorithms = {
   "Genetic Algorithm": genetic_algorithm,
    "Simulated Annealing": simulated_annealing,
    "Hill Climbing": hill_climbing
   #"DFS": dfs_n_queens
}

results = {}

for algo_name, algo_func in algorithms.items():
    results[algo_name] = []

    for N in Ns:
        if algo_name == "DFS" and N > 500:
            print(f"\nSkipping DFS for N={N} (too expensive)")
            results[algo_name].append((N, None, None))
            continue

        t, m = run_algo(algo_name, algo_func, N)
        results[algo_name].append((N, t, m))

In [ ]:
!pip install pandas
import matplotlib.pyplot as plt
import pandas as pd

# ---------- TABLE FUNCTION ----------
def create_result_table(algo_name, data):

    table_data = []

    # theoretical complexities
    complexities = {
        "DFS": "O(N!)",
        "Hill Climbing": "O(N²)",
        "Simulated Annealing": "O(N²)",
        "Genetic Algorithm": "O(P × G × N²)"
    }

    for N, t, m in data:

        table_data.append({
            "N": N,
            "Execution Time (s)": t,
            "Memory Used (MB)": m,
            "Time Complexity": complexities.get(algo_name, "Unknown")
        })

    df = pd.DataFrame(table_data)

    print(f"\n{algo_name} Results Table")
    print(df)

    return df


# ---------- GRAPH + TABLE ----------
for algo, data in results.items():

    print("\n=================================")
    print(f"Approach: {algo}")
    print("=================================")

    N_values = []
    time_values = []
    memory_values = []

    for N, t, m in data:

        print(f"N={N}, Time={t}, Memory={m}")

        # skip None values
        if t is not None and m is not None:
            N_values.append(N)
            time_values.append(t)
            memory_values.append(m)

    # ---------- CREATE TABLE ----------
    create_result_table(algo, data)

    # ---------- TIME GRAPH ----------
    plt.figure(figsize=(8, 5))

    plt.plot(
        N_values,
        time_values,
        marker='o',
        label=algo
    )

    plt.xlabel("Board Size (N)")
    plt.ylabel("Execution Time (seconds)")
    plt.title(f"{algo} Approach - Time Analysis")
    plt.legend()
    plt.grid(True)

    plt.show()

    # ---------- MEMORY GRAPH ----------
    plt.figure(figsize=(8, 5))

    plt.plot(
        N_values,
        memory_values,
        marker='o',
        label=algo
    )

    plt.xlabel("Board Size (N)")
    plt.ylabel("Memory Usage (MB)")
    plt.title(f"{algo} Approach - Memory Analysis")
    plt.legend()
    plt.grid(True)

    plt.show()